# Character LoRA Trainer (Dark Deco video)
Run on a **T4 GPU** (Runtime → Change runtime type). Steps:
1. Cell 1-2: install + config
2. Cell 3: generate hero candidates → **pick one per character** (Cell 4)
3. Cell 5: generate ~30 consistent variations per character → **delete bad ones** (Cell 6)
4. Cell 7: train (about 30-50 min per character on a T4)
5. Cell 8: LoRAs saved to `/content/char_loras` (+ zip). Upload that folder to the main notebook's runtime before it generates images (or leave it in the same Colab session).


In [ ]:
!pip -q install diffusers transformers accelerate peft bitsandbytes safetensors sentencepiece protobuf
import diffusers; print(diffusers.__version__)

In [ ]:
import json, os, random, subprocess, shutil, glob
from pathlib import Path
import torch
from PIL import Image
CHARACTERS = {
  "lead_actor": {
    "name": "The Father",
    "tokens": "45-year-old man, angular jawline, swept-back dark hair with grey temples, thick dark-rimmed glasses, charcoal three-piece suit"
  },
  "tito": {
    "name": "Tito (Son)",
    "tokens": "10-year-old boy, short black hair, round face, dark navy sweater, silver medal on ribbon around neck"
  },
  "leo": {
    "name": "Leo (Winner)",
    "tokens": "10-year-old boy, neat side-parted hair, grey blazer and white shirt, holding gold trophy"
  },
  "mammon": {
    "name": "Cartoonist Mammon",
    "tokens": "1913 cartoonist, thin mustache, rolled sleeves, ink-stained hands"
  }
}
STYLE = ", 1990s dark deco noir animation cel, hand-drawn heavy black ink outlines, flat cel shading, stark shadows"
NEG = "photorealistic, 3d render, anime big eyes, deformed hands, text, letters, watermark, blurry, multiple people"
def trigger(cid): return "ohwx" + cid.replace("_", "")
DATA = Path("/content/lora_data"); DATA.mkdir(exist_ok=True)
OUT = Path("/content/char_loras"); OUT.mkdir(exist_ok=True)
from diffusers import StableDiffusionXLPipeline, AutoencoderKL, DPMSolverMultistepScheduler
vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
pipe = StableDiffusionXLPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", vae=vae, torch_dtype=torch.float16, variant="fp16").to("cuda")
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True)
pipe.vae.enable_tiling()
def grid(paths, cols=4, size=256):
    import matplotlib.pyplot as plt
    rows = (len(paths)+cols-1)//cols
    fig, ax = plt.subplots(rows, cols, figsize=(cols*3, rows*3.3))
    ax = ax.flatten() if hasattr(ax, "flatten") else [ax]
    for a in ax: a.axis("off")
    for i, p in enumerate(paths):
        ax[i].imshow(Image.open(p)); ax[i].set_title(str(i), fontsize=14)
    plt.show()

## Stage 1: hero candidates (8 per character)

In [ ]:
HERO_DIR = Path("/content/heroes"); HERO_DIR.mkdir(exist_ok=True)
for cid, c in CHARACTERS.items():
    d = HERO_DIR / cid; d.mkdir(exist_ok=True)
    ps = []
    for i in range(8):
        g = torch.Generator("cuda").manual_seed(1000 + i)
        img = pipe(f"waist-up portrait, front facing, plain dark background, {c['tokens']}" + STYLE, negative_prompt=NEG,
                   width=832, height=1216, num_inference_steps=28, guidance_scale=7, generator=g).images[0]
        p = d / f"hero_{i}.png"; img.save(p); ps.append(p)
    print(cid, c["name"]); grid(ps)

## Pick ONE hero per character (the number above each picture)

In [ ]:
HERO = {"lead_actor": 0, "tito": 0, "leo": 0, "mammon": 0}   # <-- EDIT these numbers
print(HERO)

## Stage 2: ~30 consistent variations per character (uses your hero as an identity reference)

In [ ]:
pipe.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models", weight_name="ip-adapter_sdxl.bin")
pipe.set_ip_adapter_scale(0.75)
VARS = ["front view, neutral", "three-quarter view looking left", "three-quarter view looking right", "side profile",
 "smiling gently", "sad, eyes down", "angry, furrowed brow", "surprised, wide eyes", "thoughtful, hand on chin", "speaking, mouth open",
 "close-up face", "waist-up, arms crossed", "full body standing", "low angle", "high angle", "looking at the viewer",
 "in dim corridor with window light", "under a spotlight", "in a dark study", "outdoors at dusk, rain"]
for cid, c in CHARACTERS.items():
    d = DATA / cid; d.mkdir(exist_ok=True)
    for f in d.glob("*.png"): f.unlink()
    ref = Image.open(HERO_DIR / cid / f"hero_{HERO[cid]}.png").convert("RGB")
    ps = []
    for i in range(30):
        g = torch.Generator("cuda").manual_seed(5000 + i)
        v = VARS[i % len(VARS)]
        img = pipe(f"{c['tokens']}, {v}" + STYLE, negative_prompt=NEG, ip_adapter_image=ref,
                   width=832, height=1216, num_inference_steps=26, guidance_scale=7, generator=g).images[0]
        p = d / f"img_{i:02d}.png"; img.save(p); ps.append(p)
    print(cid, c["name"]); grid(ps, cols=6)

## Delete the images where the face/outfit looks different (list their numbers). The cleaner this set, the better the LoRA.

In [ ]:
DELETE = {"lead_actor": [], "tito": [], "leo": [], "mammon": []}   # <-- EDIT, e.g. [3, 7, 12]
for cid, idxs in DELETE.items():
    for i in idxs:
        f = DATA / cid / f"img_{i:02d}.png"
        if f.exists(): f.unlink()
    print(cid, "kept", len(list((DATA/cid).glob("*.png"))))
pipe.unload_ip_adapter(); del pipe; torch.cuda.empty_cache()

## Stage 3: train one LoRA per character (SDXL DreamBooth-LoRA). ~30-50 min each on T4.

In [ ]:
STEPS = 1000
ver = diffusers.__version__
url = f"https://raw.githubusercontent.com/huggingface/diffusers/v{ver}/examples/dreambooth/train_dreambooth_lora_sdxl.py"
subprocess.run(["wget", "-q", "-O", "train_lora_sdxl.py", url], check=True)
for cid, c in CHARACTERS.items():
    out = Path(f"/content/lora_out/{cid}"); out.mkdir(parents=True, exist_ok=True)
    cmd = ["accelerate", "launch", "train_lora_sdxl.py",
      "--pretrained_model_name_or_path=stabilityai/stable-diffusion-xl-base-1.0",
      "--pretrained_vae_model_name_or_path=madebyollin/sdxl-vae-fp16-fix",
      f"--instance_data_dir={DATA/cid}", f"--output_dir={out}",
      f"--instance_prompt={trigger(cid)}, {c['tokens']}",
      "--resolution=768", "--train_batch_size=1", "--gradient_accumulation_steps=4", "--gradient_checkpointing",
      "--use_8bit_adam", "--mixed_precision=fp16", "--learning_rate=1e-4", "--lr_scheduler=constant", "--lr_warmup_steps=0",
      f"--max_train_steps={STEPS}", "--rank=16", "--seed=0", "--center_crop"]
    print("Training", cid); subprocess.run(cmd, check=True)
    shutil.copy(out / "pytorch_lora_weights.safetensors", OUT / f"{cid}.safetensors")
print(os.listdir(OUT))

## Done: download the LoRAs

In [ ]:
shutil.make_archive("/content/char_loras", "zip", OUT)
from google.colab import files
files.download("/content/char_loras.zip")